# 03 - Cleaning the quality data

This notebook cleans the eight quality-control tables (cap, bottle, and
ink -- each with variable inspection, attribute inspection, and lot
disposition), links every inspection back to the production `LotId`,
and calculates control limits and Cp/Cpk for each characteristic.


In [1]:
import re
import pandas as pd
import numpy as np
import etl_lib as etl

RAW = "../../datasets/raw"
DIM = "../../datasets/dim"
PROCESSED = "../../datasets/processed"


## Step 1: link each inspection to the production `LotId`

The already-cleaned production table (`fact_production_processed.csv`,
from notebook `02`) has the `LotId` for every work order. Since
`WorkOrder` is a unique code across the whole plant, I can use it as a
key to link each quality inspection to the right `LotId`.

I build a `clean_and_link` function that does the basic cleaning of a
quality table and already adds the `LotId` column, so I don't repeat
this code across the 8 tables.


In [2]:
production = pd.read_csv(f"{PROCESSED}/fact_production_processed.csv", encoding="utf-8-sig")

# a lookup that, given a WorkOrder, returns the LotId for that order
work_order_to_lotid = production.set_index("WorkOrder")["LotId"]


def clean_and_link(file_name, category_columns=()):
    df = pd.read_csv(f"{RAW}/{file_name}", encoding="utf-8-sig")

    df = etl.clean_disguised_blanks(df)
    df = etl.strip_extra_spaces(df)
    df = etl.standardize_categories(df, list(category_columns))
    df, removed_count = etl.drop_duplicate_rows(df)

    # .map() looks up each WorkOrder in the work_order_to_lotid lookup
    # and brings back the matching LotId
    df["LotId"] = df["WorkOrder"].map(work_order_to_lotid)

    match_rate = df["LotId"].notna().mean()
    print(f"[{file_name}] {removed_count} duplicates removed, "
          f"{len(df):,} rows, {100*match_rate:.1f}% linked to a LotId")

    return df


## Step 2: caps (Injection Molding)

For caps, the sampling subgroup size is 10 (`M1` to `M10`), so I use
the SPC constants for n=10.


In [3]:
cap_variable = clean_and_link("fact_cap_inspection_variable_cq_raw.csv", ["Shift"])
cap_attribute = clean_and_link("fact_cap_attribute_inspection_cq_raw.csv", ["Shift"])
cap_disposition = clean_and_link("fact_cap_disposition_lot_cq_raw.csv", ["Shift"])

# grouped by Characteristic x Machine x Mold x Product -- the natural
# "one process stream" grain for a control chart (mixing different
# products on the same chart would hide real shifts behind noise)
cap_group_columns = ["Characteristic", "MachineId", "MoldId", "CapId"]

cap_variable = etl.compute_control_limits(cap_variable, cap_group_columns, subgroup_size=10)
cap_variable = etl.compute_process_capability(cap_variable, cap_group_columns, subgroup_size=10)
cap_attribute = etl.compute_attribute_indicators(cap_attribute)

cap_variable[cap_variable["Characteristic"] == "Weight"][
    ["MachineId", "MoldId", "Cp", "Cpk"]
].drop_duplicates(["MachineId", "MoldId"]).head(6)


[fact_cap_inspection_variable_cq_raw.csv] 0 duplicates removed, 83,947 rows, 100.0% linked to a LotId
[fact_cap_attribute_inspection_cq_raw.csv] 0 duplicates removed, 9,924 rows, 100.0% linked to a LotId
[fact_cap_disposition_lot_cq_raw.csv] 0 duplicates removed, 1,654 rows, 100.0% linked to a LotId


,MachineId,MoldId,Cp,Cpk
0,IM-001,M-INJ-001,1.682371,1.679875
78,IM-002,M-INJ-002,1.657785,1.649657
115,IM-003,M-INJ-003,1.682614,1.681987
181,IM-004,M-INJ-004,1.667784,1.663906
234,IM-005,M-INJ-008,1.653877,1.649078
268,IM-005,M-INJ-005,1.653657,1.648939


## Step 3: bottles (Blow Molding)

Here the subgroup size is 5, so I use the n=5 constants instead.


In [4]:
bottle_variable = clean_and_link("fact_bottle_inspection_variables_cq_raw.csv")
bottle_attribute = clean_and_link("fact_bottle_attribute_inspection_cq_raw.csv")
bottle_disposition = clean_and_link("fact_bottle_disposition_lot_cq_raw.csv", ["Shift"])

bottle_group_columns = ["Characteristic", "MachineId", "MoldId", "BottleId"]
bottle_variable = etl.compute_control_limits(bottle_variable, bottle_group_columns, subgroup_size=5)
bottle_variable = etl.compute_process_capability(bottle_variable, bottle_group_columns, subgroup_size=5)
bottle_attribute = etl.compute_attribute_indicators(bottle_attribute)

bottle_variable[bottle_variable["Characteristic"] == "Weight"][
    ["MachineId", "MoldId", "Cp", "Cpk"]
].drop_duplicates(["MachineId", "MoldId"]).head(6)


[fact_bottle_inspection_variables_cq_raw.csv] 0 duplicates removed, 124,580 rows, 100.0% linked to a LotId
[fact_bottle_attribute_inspection_cq_raw.csv] 0 duplicates removed, 17,520 rows, 100.0% linked to a LotId
[fact_bottle_disposition_lot_cq_raw.csv] 0 duplicates removed, 2,190 rows, 100.0% linked to a LotId


,MachineId,MoldId,Cp,Cpk
0,ISBM-001,M-SOP-007,1.691777,1.669219
28,ISBM-001,M-SOP-001,1.663060,1.652927
111,ISBM-002,M-SOP-008,1.648999,1.637570
272,ISBM-003,M-SOP-009,1.722986,1.715924
387,ISBM-004,M-SOP-029,1.664006,1.645975
410,ISBM-004,M-SOP-026,1.666969,1.662863


## Step 4: ink (Screen Printing)

Quality control for screen printing at this plant is 100%
attribute-based (pass/fail visual checks -- registration, coverage,
color, adhesion), with no dimensional measurement. That's why there's
no "variable inspection" table for ink, and no control chart here.


In [5]:
ink_attribute = clean_and_link("fact_ink_attribute_inspection_cq_raw.csv", ["Shift"])
ink_disposition = clean_and_link("fact_ink_disposition_lot_cq_raw.csv", ["Shift"])

ink_attribute = etl.compute_attribute_indicators(ink_attribute)

ink_attribute[["Characteristic", "ColorCount", "DefectsFound", "DefectRateP", "LotDecision"]].head()


[fact_ink_attribute_inspection_cq_raw.csv] 0 duplicates removed, 4,456 rows, 100.0% linked to a LotId
[fact_ink_disposition_lot_cq_raw.csv] 0 duplicates removed, 557 rows, 100.0% linked to a LotId


,Characteristic,ColorCount,DefectsFound,DefectRateP,LotDecision
0,Registration,2,0,0.000000,Approved
1,Centering,2,1,0.003175,Approved
2,Coverage,2,0,0.000000,Approved
3,Color,2,0,0.000000,Approved
4,Smudge,2,0,0.000000,Approved


## Step 5: save the quality tables

In [6]:
tables_to_save = {
    "fact_cap_inspection_variable_cq_processed": cap_variable,
    "fact_cap_attribute_inspection_cq_processed": cap_attribute,
    "fact_cap_disposition_lot_cq_processed": cap_disposition,
    "fact_bottle_inspection_variables_cq_processed": bottle_variable,
    "fact_bottle_attribute_inspection_cq_processed": bottle_attribute,
    "fact_bottle_disposition_lot_cq_processed": bottle_disposition,
    "fact_ink_attribute_inspection_cq_processed": ink_attribute,
    "fact_ink_disposition_lot_cq_processed": ink_disposition,
}

for name, table in tables_to_save.items():
    table.to_csv(f"{PROCESSED}/{name}.csv", encoding="utf-8-sig", index=False)

print("Quality tables saved:", list(tables_to_save.keys()))


Quality tables saved: ['fact_cap_inspection_variable_cq_processed', 'fact_cap_attribute_inspection_cq_processed', 'fact_cap_disposition_lot_cq_processed', 'fact_bottle_inspection_variables_cq_processed', 'fact_bottle_attribute_inspection_cq_processed', 'fact_bottle_disposition_lot_cq_processed', 'fact_ink_attribute_inspection_cq_processed', 'fact_ink_disposition_lot_cq_processed']


## Step 6: build the dimension tables

A well-organized data model ("star schema") needs clean dimension
tables (one row per machine, per mold, per operator) instead of
repeating this information on every fact-table row. I derive six
dimension tables from the already-clean data:

- `dim_machine` -- one row per physical machine
- `dim_mold` -- one row per physical mold/tool
- `dim_operator` -- one row per person (operator or maintenance technician)
- `dim_cap`, `dim_bottle`, `dim_ink` -- specification of each product


In [7]:
machine_capacity = pd.read_csv(f"{DIM}/dim_machine_setup.csv", encoding="utf-8-sig")
raw_cap = pd.read_csv(f"{DIM}/dim_cap.csv", encoding="utf-8-sig")
masterbatch = pd.read_csv(f"{DIM}/dim_masterbatch.csv", encoding="utf-8-sig")

# dim_machine: one row per machine
dim_machine = (
    production[["MachineId", "Process"]]
    .drop_duplicates()
    .sort_values(["Process", "MachineId"])
    .reset_index(drop=True)
)
# extracts just the number from MachineId ("IM-004" -> 4) using split instead of regex
dim_machine["MachineNumber"] = dim_machine["MachineId"].apply(lambda m: int(m.split("-")[-1]))


In [8]:
# dim_mold: one row per mold/tool, with its rated capacity
dim_mold = (
    production[["ToolId", "MachineId", "Process"]]
    .drop_duplicates()
    .rename(columns={"ToolId": "MoldId"})
)
dim_mold = dim_mold.merge(
    machine_capacity[["MoldId", "Cavities", "RatedCapacityPerHour", "CyclesPerHour", "IdealCycleTimeSec"]],
    on="MoldId", how="left"
)


In [9]:
# dim_operator: one row per person (production operator OR maintenance technician)
downtime = pd.read_csv(f"{PROCESSED}/fact_downtime_processed.csv", encoding="utf-8-sig")

operators = production[["OperatorId", "Process"]].rename(columns={"OperatorId": "PersonId"})
operators["Role"] = "Operator"

technicians = downtime.loc[
    downtime["MaintenanceTechnician"] != "N/A (Operations)",
    ["MaintenanceTechnician", "MaintenanceTeam"]
].rename(columns={"MaintenanceTechnician": "PersonId", "MaintenanceTeam": "Process"})
technicians["Role"] = "Maintenance"

people = pd.concat([operators, technicians], ignore_index=True).dropna(subset=["PersonId"])
dim_operator = people.drop_duplicates(subset=["PersonId", "Role"]).reset_index(drop=True)


In [10]:
# dim_bottle: bottle specification (volume in ml, taken from the end of ProductId)
dim_bottle = masterbatch[masterbatch["ProductType"] == "Bottle"].copy()

def extract_volume(product_id):
    # ProductId ends with the volume in ml, like "...-500"
    number = re.search(r"(\d+)$", str(product_id))
    return int(number.group(1))

dim_bottle["VolumeMl"] = dim_bottle["ProductId"].apply(extract_volume)

# dim_ink: one row per printing tool used in screen printing
screen_printing = production[production["Process"] == "Screen Printing"]
dim_ink = screen_printing[["ToolId", "ProductId"]].drop_duplicates().rename(columns={"ToolId": "PrintToolId"})

def extract_color_count(print_tool_id):
    # the tool code ends with something like "...-3C" (3 colors)
    number = re.search(r"(\d+)C$", str(print_tool_id))
    return int(number.group(1)) if number else None

dim_ink["ColorCount"] = dim_ink["PrintToolId"].apply(extract_color_count)


In [11]:
dim_machine.to_csv(f"{PROCESSED}/dim_machine.csv", encoding="utf-8-sig", index=False)
dim_mold.to_csv(f"{PROCESSED}/dim_mold.csv", encoding="utf-8-sig", index=False)
dim_operator.to_csv(f"{PROCESSED}/dim_operator.csv", encoding="utf-8-sig", index=False)
raw_cap.to_csv(f"{PROCESSED}/dim_cap.csv", encoding="utf-8-sig", index=False)
dim_bottle.to_csv(f"{PROCESSED}/dim_bottle.csv", encoding="utf-8-sig", index=False)
dim_ink.to_csv(f"{PROCESSED}/dim_ink.csv", encoding="utf-8-sig", index=False)

dimension_tables = {
    "dim_machine": dim_machine, "dim_mold": dim_mold, "dim_operator": dim_operator,
    "dim_cap": raw_cap, "dim_bottle": dim_bottle, "dim_ink": dim_ink,
}
for name, table in dimension_tables.items():
    print(f"{name}: {table.shape}")


dim_machine: (18, 3)
dim_mold: (215, 7)
dim_operator: (18, 3)
dim_cap: (31, 15)
dim_bottle: (101, 10)
dim_ink: (99, 3)
